# 2D Serial vs MPI on Sample Data

This notebook runs the sample-data helper script `ex_2d_serial_vs_mpi_sample_data.py` with real MPI, then reloads the saved rank-0 outputs into ordinary Python dictionaries so the serial and MPI results can be inspected side by side.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "examples" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Real MPI must be launched outside the notebook kernel.
# This helper script runs the sample-data case under mpirun and writes
# rank-0 outputs to disk so the notebook can load them back cleanly.
cmd = [
    "mpirun",
    "-launcher",
    "fork",
    "-n",
    "4",
    sys.executable,
    str(ROOT / "examples/python_scripts/ex_2d_serial_vs_mpi_sample_data.py"),
    "--output-dir",
    str(OUT),
]
completed = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, check=True)

summary_lines = [
    line
    for line in completed.stdout.splitlines()
    if line.startswith("MPI ranks participating:")
    or line.startswith("Saved outputs to")
    or line.startswith("Largest default diff =")
    or line.startswith("Largest full diff =")
]
print("\n".join(summary_lines))

In [ ]:
npz_data = np.load(OUT / "ex_2d_serial_vs_mpi_sample_data.npz")

# Rebuild plain dictionaries so the serial and MPI results look like the
# familiar generate_structure_functions_2d outputs inside the notebook.
default_serial_sf = {
    "x-diffs": npz_data["x_diffs_default"],
    "y-diffs": npz_data["y_diffs_default"],
    "SF_advection_velocity_x": npz_data["default_serial_SF_advection_velocity_x"],
    "SF_advection_velocity_y": npz_data["default_serial_SF_advection_velocity_y"],
}
default_mpi_sf = {
    "x-diffs": npz_data["x_diffs_default"],
    "y-diffs": npz_data["y_diffs_default"],
    "SF_advection_velocity_x": npz_data["default_mpi_SF_advection_velocity_x"],
    "SF_advection_velocity_y": npz_data["default_mpi_SF_advection_velocity_y"],
}
full_serial_sf = {
    "x-diffs": npz_data["x_diffs_full"],
    "y-diffs": npz_data["y_diffs_full"],
    "SF_advection_velocity_x": npz_data["full_serial_SF_advection_velocity_x"],
    "SF_advection_velocity_y": npz_data["full_serial_SF_advection_velocity_y"],
    "SF_advection_scalar_x": npz_data["full_serial_SF_advection_scalar_x"],
    "SF_advection_scalar_y": npz_data["full_serial_SF_advection_scalar_y"],
    "SF_LLL_x": npz_data["full_serial_SF_LLL_x"],
    "SF_LLL_y": npz_data["full_serial_SF_LLL_y"],
    "SF_LL_x": npz_data["full_serial_SF_LL_x"],
    "SF_LL_y": npz_data["full_serial_SF_LL_y"],
    "SF_LTT_x": npz_data["full_serial_SF_LTT_x"],
    "SF_LTT_y": npz_data["full_serial_SF_LTT_y"],
    "SF_LSS_x": npz_data["full_serial_SF_LSS_x"],
    "SF_LSS_y": npz_data["full_serial_SF_LSS_y"],
}
full_mpi_sf = {
    "x-diffs": npz_data["x_diffs_full"],
    "y-diffs": npz_data["y_diffs_full"],
    "SF_advection_velocity_x": npz_data["full_mpi_SF_advection_velocity_x"],
    "SF_advection_velocity_y": npz_data["full_mpi_SF_advection_velocity_y"],
    "SF_advection_scalar_x": npz_data["full_mpi_SF_advection_scalar_x"],
    "SF_advection_scalar_y": npz_data["full_mpi_SF_advection_scalar_y"],
    "SF_LLL_x": npz_data["full_mpi_SF_LLL_x"],
    "SF_LLL_y": npz_data["full_mpi_SF_LLL_y"],
    "SF_LL_x": npz_data["full_mpi_SF_LL_x"],
    "SF_LL_y": npz_data["full_mpi_SF_LL_y"],
    "SF_LTT_x": npz_data["full_mpi_SF_LTT_x"],
    "SF_LTT_y": npz_data["full_mpi_SF_LTT_y"],
    "SF_LSS_x": npz_data["full_mpi_SF_LSS_x"],
    "SF_LSS_y": npz_data["full_mpi_SF_LSS_y"],
}

default_keys = [
    "SF_advection_velocity_x",
    "SF_advection_velocity_y",
]
full_keys = [
    "SF_advection_velocity_x",
    "SF_advection_velocity_y",
    "SF_advection_scalar_x",
    "SF_advection_scalar_y",
    "SF_LLL_x",
    "SF_LLL_y",
    "SF_LL_x",
    "SF_LL_y",
    "SF_LTT_x",
    "SF_LTT_y",
    "SF_LSS_x",
    "SF_LSS_y",
]

print("Default case")
print(f"{'Structure function':<30} {'Max abs diff':>15}")
print("-" * 47)
for key in default_keys:
    diff = np.max(np.abs(default_serial_sf[key] - default_mpi_sf[key]))
    print(f"{key:<30} {diff:>15.3e}")

print("\nFull case")
print(f"{'Structure function':<30} {'Max abs diff':>15}")
print("-" * 47)
for key in full_keys:
    diff = np.max(np.abs(full_serial_sf[key] - full_mpi_sf[key]))
    print(f"{key:<30} {diff:>15.3e}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].semilogx(
    default_serial_sf["x-diffs"],
    default_serial_sf["SF_advection_velocity_x"],
    color="C0",
    linestyle="-",
    label="ASF_V x serial",
)
axes[0].semilogx(
    default_mpi_sf["x-diffs"],
    default_mpi_sf["SF_advection_velocity_x"],
    color="C0",
    linestyle="--",
    label="ASF_V x mpi",
)
axes[0].semilogx(
    default_serial_sf["y-diffs"],
    default_serial_sf["SF_advection_velocity_y"],
    color="C1",
    linestyle="-",
    label="ASF_V y serial",
)
axes[0].semilogx(
    default_mpi_sf["y-diffs"],
    default_mpi_sf["SF_advection_velocity_y"],
    color="C1",
    linestyle="--",
    label="ASF_V y mpi",
)

axes[1].semilogx(
    full_serial_sf["x-diffs"],
    full_serial_sf["SF_LL_x"],
    color="C2",
    linestyle="-",
    label="LL x serial",
)
axes[1].semilogx(
    full_mpi_sf["x-diffs"],
    full_mpi_sf["SF_LL_x"],
    color="C2",
    linestyle="--",
    label="LL x mpi",
)
axes[1].semilogx(
    full_serial_sf["y-diffs"],
    full_serial_sf["SF_LLL_y"],
    color="C3",
    linestyle="-",
    label="LLL y serial",
)
axes[1].semilogx(
    full_mpi_sf["y-diffs"],
    full_mpi_sf["SF_LLL_y"],
    color="C3",
    linestyle="--",
    label="LLL y mpi",
)

axes[2].semilogx(
    full_serial_sf["x-diffs"],
    full_serial_sf["SF_LTT_x"],
    color="C4",
    linestyle="-",
    label="LTT x serial",
)
axes[2].semilogx(
    full_mpi_sf["x-diffs"],
    full_mpi_sf["SF_LTT_x"],
    color="C4",
    linestyle="--",
    label="LTT x mpi",
)
axes[2].semilogx(
    full_serial_sf["y-diffs"],
    full_serial_sf["SF_LSS_y"],
    color="C5",
    linestyle="-",
    label="LSS y serial",
)
axes[2].semilogx(
    full_mpi_sf["y-diffs"],
    full_mpi_sf["SF_LSS_y"],
    color="C5",
    linestyle="--",
    label="LSS y mpi",
)

for ax in axes:
    ax.set_xlabel("Separation distance")
    ax.set_ylabel("Structure function")
    ax.legend(fontsize=7)

fig.tight_layout()
fig